In [1]:
import pandas as pd
import numpy as np
import calendar
from pathlib import Path

root = Path('.')
var = 't2m' # Variable name: ['t2m', 'ssr', 'sf', 'rf']

In [9]:
# Calculate the daily climatology and anomalies for the selected ERA5 variable
# Remove leap-day records to keep a consistent 365-day climatology

var_data = pd.read_csv(root / f'{var}_extraction.csv', index_col=0)
var_data.index = pd.to_datetime(var_data.index, format="%Y-%m-%d")

leap_dates = pd.to_datetime([f"{year}-02-29" for year in range(1951, 2023) if calendar.isleap(year)])
leap_dates = leap_dates.strftime('%Y-%m-%d')
var_data.drop(index=leap_dates, inplace=True)

month_day = var_data.index.strftime('%m-%d')
daily_mean = var_data.groupby(month_day).transform('mean')
daily_mean.to_csv(root / f'{var}_clim_mean.csv')
df_anomaly = var_data - daily_mean
df_anomaly.to_csv(root / f'{var}_anom.csv')

print(df_anomaly.head())

                  10       100       102       104       105       106  \
1950-09-01  1.118116 -1.120435  1.868600  0.056905 -1.154891 -0.613643   
1950-09-02 -0.829354  0.181629 -0.850010  0.042502  0.874008 -0.069518   
1950-09-03 -4.228733 -2.547946 -2.917727 -2.756971 -1.472263 -2.673276   
1950-09-04 -3.690848 -3.901299 -3.396414 -4.001209 -4.270673 -3.426830   
1950-09-05 -3.481910 -4.178949 -3.434828 -3.815318 -4.394777 -4.106409   

                 108       118       119       123  ...    xNL095    xNL096  \
1950-09-01 -0.349105 -1.243571 -1.243571 -0.630008  ... -1.849385 -0.989068   
1950-09-02 -0.241827  0.654847  0.654847  0.354430  ... -0.218937 -0.272943   
1950-09-03 -2.802559 -2.216231 -2.216231 -3.186176  ...  0.697327 -0.911714   
1950-09-04 -3.260292 -3.953813 -3.953813 -4.182055  ...  2.111479  0.233990   
1950-09-05 -4.235821 -4.762086 -4.762086 -4.247266  ...  2.174752  0.618261   

              xNL098    xNL099    xNL100    xNL101    xNL102     xSS01  \
1950-0

In [7]:
# Extract AR-associated anomalies

var_anoms = pd.read_csv(root / f'{var}_anom.csv', index_col=0)
var_anoms.index = pd.to_datetime(var_anoms.index, format="%Y-%m-%d")
ars = pd.read_csv(root / 'AR_extraction_daily.csv', index_col=0)
ars.index = pd.to_datetime(ars.index, format="%Y-%m-%d")
ars.drop(index=leap_dates, inplace=True)

ar_var_anom = var_anoms * ars
ar_var_anom.to_csv(root / f'AR_related_{var}_anom.csv')
print(ar_var_anom.head())

                  10       100     102       104       105       106  \
1950-09-01  1.118116 -1.120435  1.8686  0.056905 -1.154891 -0.613643   
1950-09-02 -0.000000  0.000000 -0.0000  0.000000  0.000000 -0.000000   
1950-09-03 -0.000000 -0.000000 -0.0000 -0.000000 -0.000000 -0.000000   
1950-09-04 -0.000000 -0.000000 -0.0000 -0.000000 -0.000000 -0.000000   
1950-09-05 -0.000000 -0.000000 -0.0000 -0.000000 -0.000000 -0.000000   

                 108       118       119       123  ...    xNL095    xNL096  \
1950-09-01 -0.349105 -1.243571 -1.243571 -0.630008  ... -0.000000 -0.000000   
1950-09-02 -0.000000  0.000000  0.000000  0.000000  ... -0.000000 -0.000000   
1950-09-03 -0.000000 -0.000000 -0.000000 -0.000000  ...  0.000000 -0.000000   
1950-09-04 -0.000000 -0.000000 -0.000000 -0.000000  ...  0.000000  0.000000   
1950-09-05 -0.000000 -0.000000 -0.000000 -0.000000  ...  2.174752  0.618261   

             xNL098    xNL099  xNL100    xNL101    xNL102  xSS01    xTAK01  \
1950-09-01 -0.

In [8]:
# Aggregate AR-associated anomalies by hydrological year and by season

def calcu_seasonal_anom(var):
    ar_var_anom = pd.read_csv(root / f'AR_related_{var}_anom.csv', index_col=0)
    ar_var_anom.index = pd.to_datetime(ar_var_anom.index)
    
    ar_var_anom.replace(0, np.nan, inplace=True)
    
    hydro_year = ar_var_anom.index.year.copy()
    hydro_year = pd.Series(hydro_year, index=ar_var_anom.index)
    hydro_year[ar_var_anom.index.month >= 9] += 1
    
    autumn_winter_mask = (ar_var_anom.index.month >= 9) | (ar_var_anom.index.month <= 2)
    spring_summer_mask = (ar_var_anom.index.month >= 3) & (ar_var_anom.index.month <= 8)
    
    annual_data = ar_var_anom.groupby(hydro_year).mean()
    autumn_winter_data = ar_var_anom[autumn_winter_mask].groupby(hydro_year).mean()
    spring_summer_data = ar_var_anom[spring_summer_mask].groupby(hydro_year).mean()
    
    if var in ['sf', 'rf']:
        annual_data = annual_data * 1000
        autumn_winter_data = autumn_winter_data * 1000
        spring_summer_data = spring_summer_data * 1000
    
    if var in ['ssr']:
        annual_data = annual_data / 86400
        autumn_winter_data = autumn_winter_data / 86400
        spring_summer_data = spring_summer_data / 86400
    
    annual_data.to_csv(root / f'AR_related_{var}_anom_ALLYear.csv')
    autumn_winter_data.to_csv(root / f'AR_related_{var}_anom_AutWin.csv')
    spring_summer_data.to_csv(root / f'AR_related_{var}_anom_SprSum.csv')
    print(annual_data.head())
    return 

calcu_seasonal_anom(var)

            10       100       102       104       105       106       108  \
1951  3.541226  3.052838  3.690169  2.905937  3.596433  2.895181  2.712962   
1952  3.767081  3.075225  3.551487  2.826047  3.268280  2.764583  2.762168   
1953  4.551975  4.093442  4.288026  4.007410  4.235724  3.941736  4.098272   
1954  3.993446  3.284637  4.441082  3.311029  3.614050  3.239601  3.246048   
1955  3.206987  3.041538  2.868624  3.163166  2.732932  3.022806  3.021251   

           118       119       123  ...    xNL095    xNL096    xNL098  \
1951  3.507626  3.507626  3.052553  ...  1.759922  0.339280  0.088312   
1952  3.226943  3.226943  3.146605  ...  3.032720  2.320566  2.077865   
1953  4.370758  4.370758  4.105962  ...  3.618166  2.572776  2.611491   
1954  3.578277  3.578277  3.448379  ...  2.719486  3.045103  2.955440   
1955  2.765660  2.765660  2.588393  ...  1.470968  2.629395  2.120203   

        xNL099    xNL100    xNL101    xNL102     xSS01    xTAK01    xTN001  
1951 -0.020987 